In [0]:
%run ./_bootstrap

In [0]:
import os
import json
import tempfile
import hashlib
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import mlflow
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
import joblib
from helpers.mlflow_retriever import load_retriever_artifacts

In [0]:
spark = SparkSession.builder.getOrCreate()
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA med")

In [0]:
client = MlflowClient()
exp_name = "/Users/dhrutigandhi.05@gmail.com/medibot_retrieval"
exp = client.get_experiment_by_name(exp_name) # get experiment if it already exists

# create experiment if it does not exist
if exp is None:
    exp_id = client.create_experiment(exp_name)
else:
    exp_id = exp.experiment_id

# set experiment by id
mlflow.set_experiment(experiment_id=exp_id)
print("Using experiment:", exp_name)
print("Experiment id:", exp_id)


In [0]:
TRAIN_RUN_ID = "4ea1952dd36348f997c9791e6c5fadc4" # set the mlflow run id that contains vectorizer.joblib, knn.joblib, and chunk_ids.json
vectorizer, knn, chunk_ids, train_metadata = load_retriever_artifacts(run_id=TRAIN_RUN_ID, artifact_path="retriever")

print("TRAIN_RUN_ID:", TRAIN_RUN_ID)
print("num chunk_ids:", len(chunk_ids))
print("train_metadata keys:", list(train_metadata.keys()))

In [0]:
doc_chunks_df = spark.table("workspace.med.doc_chunks") # load the chunk table from delta
serve_cols = ["chunk_id", "doc_id", "source", "category", "title", "chunk_text"] # choose the columns we want to return from the endpoint
filtered_df = doc_chunks_df.select(*serve_cols).where(F.col("chunk_id").isin(chunk_ids)) # filter to only chunk_ids in the training run
chunks_pdf = filtered_df.toPandas() # convert to pandas for serving
pos_map = {cid: i for i, cid in enumerate(chunk_ids)} # build a position map to restore the exact training order
chunks_pdf = chunks_pdf[chunks_pdf["chunk_id"].isin(pos_map)] # keep only rows that exist in pos_map

# add position and sort
chunks_pdf["__pos"] = chunks_pdf["chunk_id"].map(pos_map)
chunks_pdf = chunks_pdf.sort_values("__pos").drop(columns=["__pos"]).reset_index(drop=True)

# sanity checks to confirm alignment
print("chunks_pdf rows:", len(chunks_pdf))
print("first matches:", chunks_pdf.iloc[0]["chunk_id"] == chunk_ids[0])
print("last matches:", chunks_pdf.iloc[-1]["chunk_id"] == chunk_ids[-1])